## Create Distances = 1 - CKA

In [12]:
import os
import glob
import pandas as pd



In [13]:
def convert_cka_to_distance(cka_csv_path, output_dir):
    for csv_path in glob.glob(f'{cka_csv_path}/*optimal*.csv'):
        df = pd.read_csv(csv_path, index_col='sample_idx')
        df_distance = 1 - df
        out_path = os.path.join(output_dir, os.path.basename(csv_path))
        df_distance.to_csv(out_path)

cka_csv_path, output_dir = 'retrain_outputs_CKA', 'retrain_output_distance'
# cka_csv_path, output_dir = 'outputs_CKA', 'output_distance'
convert_cka_to_distance(cka_csv_path, output_dir)

## Draw graph by dataset by model

In [ ]:
import os
import glob
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

os.makedirs('retrain_graphs_distance', exist_ok=True)

# Group files by ticker: {ticker: {model_name: csv_path}}
files_by_ticker = defaultdict(dict)
for csv_path in glob.glob('retrain_output_distance/*.csv'):
    fname = os.path.basename(csv_path)
    m = re.match(r'(\d+Layers)_optimal_(\w+)_cka\.csv', fname)
    if m:
        model_name, ticker = m.group(1), m.group(2)
        files_by_ticker[ticker][model_name] = csv_path

model_order = ['5Layers']

for ticker, model_files in sorted(files_by_ticker.items()):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Dataset: {ticker}', fontsize=16, fontweight='bold')

    for idx, model_name in enumerate(model_order):
        ax = axes[idx // 2][idx % 2]

        if model_name not in model_files:
            ax.set_visible(False)
            continue

        df = pd.read_csv(model_files[model_name], index_col='sample_idx')
        layer_indices = list(range(1, len(df.columns) + 1))
        alpha = max(0.05, min(0.3, 30.0 / len(df)))

        for row in df.itertuples(index=False):
            ax.plot(layer_indices, list(row), alpha=alpha, linewidth=0.6, color='steelblue')

        ax.set_title(f'{model_name} Model — {ticker}')
        ax.set_xlabel('Layer Index')
        ax.set_ylabel('Distance (1 - CKA)')
        ax.set_xticks(layer_indices)
        ax.set_xticklabels([f'Layer {i}' for i in layer_indices])

    plt.tight_layout()
    plt.savefig(f'retrain_graphs_distance/{ticker}_distances_comparison_9.png', dpi=150)
    plt.close()

## Draw graph for all data per model

In [ ]:
import os
import glob
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# Group files by model: {model_name: [csv_path, ...]}
files_by_model = defaultdict(list)
for csv_path in glob.glob('retrain_output_distance/*9*optimal*.csv'):
    m = re.match(r'(\d+Layers)_optimal_\w+_cka\.csv', os.path.basename(csv_path))
    if m:
        files_by_model[m.group(1)].append(csv_path)

model_order = ['9Layers']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distance Comparison Across All Datasets', fontsize=16, fontweight='bold')

for idx, model_name in enumerate(model_order):
    ax = axes[idx // 2][idx % 2]

    frames = [pd.read_csv(p, index_col='sample_idx') for p in files_by_model[model_name]]
    df_all = pd.concat(frames, ignore_index=True)
    print(df_all.shape)

    layer_indices = list(range(1, len(df_all.columns) + 1))
    alpha = max(0.05, min(0.3, 30.0 / len(df_all)))

    for row in df_all.itertuples(index=False):
        ax.plot(layer_indices, list(row), alpha=alpha, linewidth=0.6, color='steelblue')

    ax.set_title(f'{model_name} Model')
    ax.set_xlabel('Layer Index')
    ax.set_ylabel('Distance (1 - CKA)')
    ax.set_xticks(layer_indices)
    ax.set_xticklabels([f'Layer {i}' for i in layer_indices])

plt.tight_layout()
plt.savefig('retrain_graphs_distance/01_distances_by_model_9Layers.png', dpi=150)
plt.close()

(54260, 4)
